In [ ]:
import pandas as pd
train_source1 = pd.read_csv(r"E:\ANISH\Amazon ML\student_resource\dataset\train\train_source1.tsv",
sep="\t")
train_source2 = pd.read_csv(r"E:\ANISH\Amazon ML\student_resource\dataset\train\train_source2.tsv",
sep="\t")
train_source3 = pd.read_csv(r"E:\ANISH\Amazon ML\student_resource\dataset\train\train_source3.tsv",
sep="\t")
train_ground_truth=pd.read_csv(r"E:\ANISH\Amazon ML\student_resource\dataset\train\train_ground_truth.tsv",
sep="\t")



In [ ]:
train_source1.head()

In [ ]:
train_source2.head()

In [ ]:
train_source3.head()

In [ ]:
train_ground_truth.head()

In [ ]:
train_source1.shape, train_source2.shape, train_source3.shape, train_ground_truth.shape

In [ ]:
tables = {
    "source1": train_source1,
    "source2": train_source2,
    "source3": train_source3,
    "ground_truth": train_ground_truth,
}

for name, df in tables.items():
    print(f"\n{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    print("Missing values:\n", df.isna().sum())
    id_column = "source1_entity_id" if name == "ground_truth" else "entity_id"
    print("Duplicate IDs:", df[id_column].duplicated().sum())

In [ ]:
for name, df in {
    "source1": train_source1,
    "source2": train_source2,
    "source3": train_source3,
}.items():
    print(f"\n{name} country counts:")
    print(df["country"].value_counts(dropna=False))

matched_ids = train_ground_truth["matched_entity_ids"].fillna("").str.strip()
match_counts = matched_ids.map(lambda value: len(value.split(",")) if value else 0)

print("\nNumber of matches per Source 1 entity:")
print(match_counts.value_counts().sort_index())
print("Singletons (no matches):", (match_counts == 0).sum())

In [ ]:
test=pd.read_csv(r"E:\ANISH\Amazon ML\student_resource\dataset\test\test_source1.tsv",
sep="\t")
test.head()


In [ ]:
# Make one row per Source 1-to-Source 2/3 link.
# An empty match list is kept as one row so singleton entities remain in the table.
links = train_ground_truth[
    ["source1_entity_id", "matched_entity_ids"]
].copy()

links["matched_entity_ids"] = (
    links["matched_entity_ids"]
    .fillna("")
    .str.split(",")
)

links = links.explode("matched_entity_ids", ignore_index=True)
links["matched_entity_ids"] = links["matched_entity_ids"].str.strip()

# Add details for the Source 1 record.
source1_records = train_source1.rename(columns={
    "entity_id": "source1_entity_id",
    "business_name": "source1_business_name",
    "business_address": "source1_business_address",
    "country": "source1_country",
})

# Combine Source 2 and Source 3 as possible matched records.
target_records = pd.concat(
    [
        train_source2.assign(matched_source="S2"),
        train_source3.assign(matched_source="S3"),
    ],
    ignore_index=True,
).rename(columns={
    "entity_id": "matched_entity_ids",
    "business_name": "matched_business_name",
    "business_address": "matched_business_address",
    "country": "matched_country",
})

# Attach the business details. Empty match IDs remain with empty matched columns.
training_pairs = (
    links
    .merge(source1_records, on="source1_entity_id", how="left")
    .merge(target_records, on="matched_entity_ids", how="left")
)

training_pairs = training_pairs[
    [
        "source1_entity_id",
        "source1_business_name",
        "source1_business_address",
        "source1_country",
        "matched_entity_ids",
        "matched_business_name",
        "matched_business_address",
        "matched_country",
        "matched_source",
    ]
]



In [ ]:
training_pairs.head(10)

In [ ]:
training_pairs.shape